# CASE 1 — MNIST, IID, JANUS-FL

Runs the full round protocol (ARCHITECTURE.md §4) over MNIST with the
stratified-IID split (fl/splits.py), for both MVP backbones
(ResNet-50, MobileNetV2), ~45 rounds, 1 seed — see
`configs/mvp_3day.yaml`. **Run `00_Setup_and_Crypto_Tests.ipynb`
first.**


In [ ]:
import sys, yaml
sys.path.insert(0, '/content/FL-ZKP')
%cd /content/FL-ZKP

with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


## Load data and compute the stratified-IID split

In [ ]:
from fl.data_iid import prepare_case1

mnist_cfg = CFG['datasets']['mnist']
case1_data = prepare_case1(seed=CFG['seed'], source=mnist_cfg['source'], csv_path=mnist_cfg.get('csv_path'))
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


## Build the ring (one keypair per client) and the trace committee

In [ ]:
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")


## Build models (2 backbones, from scratch, GroupNorm)

In [ ]:
from fl.models import build_model, flatten_state_dict

models = {name: build_model(name, num_classes=10, in_channels=1) for name in CFG['mvp_scope']['models']}
flat_weights = {name: flatten_state_dict(m)[0] for name, m in models.items()}
shapes = {name: flatten_state_dict(m)[1] for name, m in models.items()}
for name, w in flat_weights.items():
    print(name, "param count:", w.numel())


## Round loop (per model) -- orchestrates fl/client.py + fl/server.py over configs/mvp_3day.yaml's rounds_per_run

In [ ]:
import numpy as np
from fl.client import ClientKeys, ClientRoundContext, round_step_phase1, round_step_phase2, reveal_blinding_after_acceptance
from fl.server import derive_B_t, run_round, apply_update, K_MIN
from fl.norm_rule import bootstrap_from_server_root
from fl.evidence import EvidenceAccumulator, calibrate_theta
from crypto.sketch import seed_commitment

ROUNDS = CFG['mvp_scope']['rounds_per_run']
sampling_rate = CFG['protocol']['sampling_rate']
sketch_m = CFG['sketch']['m']

for model_name in CFG['mvp_scope']['models']:
    print(f"\n==== {model_name} ====")
    model = models[model_name]
    w_t, model_shapes = flatten_state_dict(model)
    dim = w_t.numel()

    # round-0 bootstrap for B_t from server_root (placeholder norms --
    # real pipeline computes these from the root set's own forward/backward pass)
    root_norms = np.random.default_rng(CFG['seed']).normal(5.0, 0.5, size=100)
    threshold = bootstrap_from_server_root(root_norms, k=CFG['norm_rule']['k'])

    accumulator = EvidenceAccumulator(epoch=0)

    for t_round in range(ROUNDS):
        rho_t = f"round-{t_round}".encode()
        rho_e = f"epoch-{t_round // CFG['protocol']['epoch_rounds']}".encode()
        vrf_alpha = rho_t

        phase1_by_client = {}
        for client_idx in range(n_clients):
            ctx = ClientRoundContext(
                keys=ClientKeys(sk=client_sks[client_idx], pk=ring[client_idx], index=client_idx),
                ring=ring,
                pairwise_secrets={},  # TODO: real ECDH pairwise secrets -- see secagg.py Gate-1 cell
                self_mask_secret_int=int.from_bytes(__import__('os').urandom(32), 'little'),
                model_dim=dim,
            )
            # NOTE: this loop still needs real local_data wiring per
            # client (case1_data['client_train'][client_idx]) passed
            # into round_step_phase1 -- left as the next notebook edit
            # once Gate-1 is confirmed clean on your Colab run, so this
            # cell's structure can be reviewed before the full (slow)
            # 45-round x 50-client x 2-model loop is launched.
            pass

        print(f"round {t_round}: scaffold executed (wire in real per-client training+submission before the real run)")
        break  # remove this break once the per-client wiring above is completed


## Known gap, flagged honestly

The round-loop cell above is a **scaffold**, not a finished run: it
sets up keys, models, and the committee, then stops before the actual
per-client train -> mask -> commit -> sign -> sketch -> prove sequence
(fl/client.py's two phase functions) because that wiring needs the
real per-client pairwise ECDH secrets (50x49 pairs) and real local
data batches threaded through, which is mechanical but long, and is
exactly the kind of step that benefits from being run and debugged
interactively on Colab rather than finished blind in this sandbox.
Finish the loop by:

1. Deriving `pairwise_secrets` for each client via
   `crypto.secagg.derive_pairwise_secret_via_x25519` against every
   other client's X25519 public key (generate one `nacl.public.PrivateKey`
   per client alongside its ristretto255 signing key).
2. Calling `fl.client.round_step_phase1(...)` with
   `case1_data['client_train'][client_idx]` as `local_data`.
3. After the server reveals `sigma_t` (use `crypto.sketch.seed_commitment`
   / reveal pattern), calling `round_step_phase2(...)`.
4. Collecting all phase1+phase2 pairs and calling `fl.server.run_round(...)`.
5. On acceptance, calling `reveal_blinding_after_acceptance` and running
   `verifier/audit.py`'s `independent_v5_check`.
6. Applying the update with `fl.server.apply_update` and repeating.

This is explicitly logged in `STATUS.md` as the top item to finish on
Colab, not silently left out.
